# 9.3.深度循环神经网络

到目前为止，我们只讨论了具有一个单向隐藏层的循环神经网络。 其中，隐变量和观测值与具体的函数形式的交互方式是相当随意的。 只要交互类型建模具有足够的灵活性，这就不是一个大问题。 然而，对一个单层来说，这可能具有相当的挑战性。之前在线性模型中，我们通过添加更多的层来解决这个问题。而在循环神经网络中，我们首先需要确定如何添加更多的层，以及在哪里添加额外的非线性，因此这个问题有点棘手。

事实上，我们可以将多层循环神经网络堆叠在一起，通过对几个简单层的组合，产生了一个灵活的机制。特别是，数据可能与不同层的堆叠有关。例如，我们可能希望保持有关金融市场状况 （熊市或牛市）的宏观数据可用， 而微观数据只记录较短期的时间动态。下图描述了一个具有$L$个隐藏层的深度循环神经网络， 每个隐状态都连续地传递到当前层的下一个时间步和下一层的当前时间步。

<div align="center">
  <img src="./images/deep-rnn.svg" alt="图9.3.1 深度循环神经网络结构" width="400">
  <br><small>图9.3.1 深度循环神经网络结构</small>
</div>

---
## 9.3.1.环境配置

In [1]:
%pip install pypto==0.2.0 torch torch_npu matplotlib

In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import torch_npu
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

In [3]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import PyPTOLinear, PyPTOLSTM, loss_fn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
</pre>
  </div>
</details>


---
## 9.3.2.函数依赖关系

我们可以将深度架构中的函数依赖关系形式化， 这个架构是由图中描述的$L$个隐藏层构成。后续的讨论主要集中在经典的循环神经网络模型上， 但是这些讨论也适用于其他序列模型。

假设在时间步$t$有一个小批量的输入数据 $\mathbf{X}_t \in \mathbb{R}^{n \times d}$ （样本数：$n$，每个样本中的输入数：$d$）。 同时，将$l^\mathrm{th}$隐藏层（$l=1,\ldots,L$） 的隐状态设为$\mathbf{H}_t^{(l)} \in \mathbb{R}^{n \times h}$ （隐藏单元数：$h$）， 输出层变量设为$\mathbf{O}_t \in \mathbb{R}^{n \times q}$ （输出数：$q$）。 设置$\mathbf{H}_t^{(0)} = \mathbf{X}_t$， 第$l$个隐藏层的隐状态使用激活函数$\phi_l$，则：

$$\tag{9.3.1}\mathbf{H}_t^{(l)} = \phi_l(\mathbf{H}_t^{(l-1)} \mathbf{W}_{xh}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{hh}^{(l)} + \mathbf{b}_h^{(l)}),$$

其中，权重$\mathbf{W}_{xh}^{(1)} \in \mathbb{R}^{d \times h}$，对于$l>1$时$\mathbf{W}_{xh}^{(l)} \in \mathbb{R}^{h \times h}$， $\mathbf{W}_{hh}^{(l)} \in \mathbb{R}^{h \times h}$和偏置$\mathbf{b}_h^{(l)} \in \mathbb{R}^{1 \times h}$ 都是第$l$个隐藏层的模型参数。

最后，输出层的计算仅基于第$L$个隐藏层最终的隐状态：

$$\tag{9.3.2}\mathbf{O}_t = \mathbf{H}_t^{(L)} \mathbf{W}_{hq} + \mathbf{b}_q,$$

其中，权重$\mathbf{W}_{hq} \in \mathbb{R}^{h \times q}$和偏置$\mathbf{b}_q \in \mathbb{R}^{1 \times q}$都是输出层的模型参数。

与多层感知机一样，隐藏层数目$L$和隐藏单元数目$h$都是超参数。 也就是说，它们可以由我们调整。 另外，用门控循环单元或长短期记忆网络的隐状态 来代替上述RNN中的隐状态进行计算， 可以很容易地得到深度门控循环神经网络或深度长短期记忆神经网络。

---
## 9.3.3.简洁实现

实现多层循环神经网络所需的许多逻辑细节在高级API中都是现成的。 简单起见，我们仅示范使用此类内置函数的实现方式。 以长短期记忆网络模型为例， 该代码与之前在 [9.2节](09.02_lstm.ipynb)中使用的代码非常相似， 实际上唯一的区别是我们指定了层的数量， 而不是使用单一层这个默认值。 像往常一样，我们从加载数据集开始。

像选择超参数这类架构决策也跟 [9.2节](09.02_lstm.ipynb)中的决策非常相似。 因为我们有不同的词元，所以输入和输出都选择相同数量，即`vocab_size`。 隐藏单元的数量仍然是$256$。 唯一的区别是，我们现在**通过`num_layers`的值来设定隐藏层数**。

In [4]:
vocab_size, num_hiddens = len(vocab), 256

class PyPTORNNModel(nn.Module):
    """循环神经网络语言模型：PyPTOLSTM 隐层 + PyPTOLinear 输出层。"""

    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        self.linear = PyPTOLinear(self.num_hiddens, vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(device=inputs.device, dtype=torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, batch_size, device=None):
        if device is None:
            device = next(self.parameters()).device
        return (torch.zeros(self.rnn.num_layers, batch_size, self.num_hiddens, device=device),
                torch.zeros(self.rnn.num_layers, batch_size, self.num_hiddens, device=device))

lstm_layer = PyPTOLSTM(len(vocab), num_hiddens, num_layers=2)
model = PyPTORNNModel(lstm_layer, len(vocab))
model = model.to(device)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>模型封装流程</b>：输入 <code>one_hot</code> 编码 → <code>PyPTOLSTM</code> 隐层（<code>num_layers=2</code>，PyPTOLSTM 支持多层，与原书 <code>nn.LSTM(num_inputs, num_hiddens, num_layers)</code> 对齐）→ <code>PyPTOLinear</code> 输出层映射到词表。</li>
      <li style="margin: 0 0 8px 0;"><b>多层堆叠的数据流</b>：深度 RNN 中第 <code>l</code> 层隐状态同时传向当前层的下一时间步与下一层的当前时间步（<code>H_t^(l) = φ(H_t^(l-1) W_xh + H_(t-1)^(l) W_hh + b)</code>），输出层只看最顶层隐状态。</li>
      <li style="margin: 0 0 8px 0;"><b>初始状态</b>：<code>begin_state</code> 按 <code>self.rnn.num_layers</code> 返回 <code>(H, C)</code> 全零状态，各 <code>(num_layers, batch, hidden)</code>，供截断 BPTT 跨 batch 传递（<code>train_ch8</code> 在每个 epoch 开始时重置）。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
num_inputs = vocab_size
device = d2l.try_gpu()
lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers)
model = d2l.RNNModel(lstm_layer, len(vocab))
model = model.to(device)
</pre>
  </div>
</details>


---
## 9.3.4.训练与预测

由于使用了长短期记忆网络模型来实例化两个层，因此训练速度被大大降低了。

In [5]:
# 预热：触发 PyPTO kernel（LSTM 隐层 + 输出层）的首次编译
print('正在编译 pypto kernel（首次运行耗时较长，请耐心等待）...')
# 取一个批次触发 JIT 编译
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = model.begin_state(X.shape[0], device)
Y, new_state = model(X, state)
y = y.T.reshape(-1).to(device)
# 使用 PyPTO loss_fn：softmax + CE 全程在 NPU 上执行
l = loss_fn(Y, y.long(), num_classes=len(vocab))
l.backward()
# 重置梯度，为正式训练做准备
model.zero_grad()
print('编译完成（耗时较长）。接下来可以正常训练了。')

正在编译 pypto kernel（首次运行耗时较长，请耐心等待）...


编译完成（耗时较长）。接下来可以正常训练了。


In [6]:
num_epochs, lr = 500, 2
train_ch8(model, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

困惑度 1.1, 2156.5 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e


traveller soming poomemh time tricllis that is all right sa


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_epochs, lr = 500, 2
d2l.train_ch8(model, train_iter, vocab, lr*1.0, num_epochs, device)
</pre>
  </div>
</details>


---
## 9.3.5.小结

* 在深度循环神经网络中，隐状态的信息被传递到当前层的下一时间步和下一层的当前时间步。
* 有许多不同风格的深度循环神经网络，
 如长短期记忆网络、门控循环单元、或经典循环神经网络。
 这些模型在深度学习框架的高级API中都有涵盖。
* 总体而言，深度循环神经网络需要大量的调参（如学习率和裁剪）
 来确保合适的收敛，模型的初始化也需要谨慎。

---
## 9.3.6.练习

1. 基于我们在 [8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)中讨论的单层实现，
 尝试从零开始实现两层循环神经网络。
1. 在本节训练模型中，比较使用门控循环单元替换长短期记忆网络后模型的精确度和训练速度。
1. 如果增加训练数据，能够将困惑度降到多低？
1. 在为文本建模时，是否可以将不同作者的源数据合并？有何优劣呢？


参考答案详见 [answers/09.03_reference_answer](./answers/09.03_reference_answer.ipynb)。


### 9.3.6.1.参考答案（PyPTO）

In [7]:
!cat answers/txt/09.03_reference_answer_pypto.txt

### 9.3.6.2.参考答案（PyTorch）

In [8]:
!cat answers/txt/09.03_reference_answer_pytorch.txt